# 04. Демонстрация production pipeline

Цель ноутбука — показать, как готовая production-модель загружается из `artifacts/movie_revenue_pipeline.joblib` и возвращает прогноз обычной кассовой выручки, а не `log_gross`.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PIPELINE_PATH = PROJECT_ROOT / "artifacts" / "movie_revenue_pipeline.joblib"
MODEL_INFO_PATH = PROJECT_ROOT / "artifacts" / "model_info.json"
METRICS_PATH = PROJECT_ROOT / "artifacts" / "metrics.json"
FEATURE_SCHEMA_PATH = PROJECT_ROOT / "artifacts" / "feature_schema.json"
SAMPLE_INPUT_PATH = PROJECT_ROOT / "artifacts" / "sample_input.json"

pipeline = joblib.load(PIPELINE_PATH)

with MODEL_INFO_PATH.open("r", encoding="utf-8") as file:
    model_info = json.load(file)
with METRICS_PATH.open("r", encoding="utf-8") as file:
    metrics = json.load(file)
with FEATURE_SCHEMA_PATH.open("r", encoding="utf-8") as file:
    feature_schema = json.load(file)
with SAMPLE_INPUT_PATH.open("r", encoding="utf-8") as file:
    sample_input = json.load(file)

model_info

## Метрики сохранённой модели

In [ ]:
pd.Series(metrics, name="value").to_frame()

## Схема входных признаков

`title/name` и `released` используются как metadata. В модель не должен передаваться `gross`, потому что это целевая переменная.

In [ ]:
display(pd.Series(feature_schema["model_input_columns"], name="model_input_columns"))
display(pd.Series(feature_schema["unused_columns"], name="unused_columns"))

## Пример прогноза

Pipeline уже содержит `TransformedTargetRegressor`, поэтому `pipeline.predict()` возвращает обычную прогнозируемую кассовую выручку в USD.

In [ ]:
sample_input

In [ ]:
model_input = {
    key: value
    for key, value in sample_input.items()
    if key in feature_schema["model_input_columns"]
}

input_df = pd.DataFrame([model_input])
predicted_revenue = float(max(pipeline.predict(input_df)[0], 0.0))
predicted_revenue

## Бизнес-метрики

После прогноза выручки рассчитываются показатели коммерческого успеха:

- `predicted_profit = predicted_revenue - budget`;
- `roi = predicted_profit / budget`;
- `payback_ratio = predicted_revenue / budget`.

In [ ]:
def classify_success(roi):
    if roi < 0:
        return "коммерчески неуспешный"
    if roi < 1:
        return "умеренно успешный"
    if roi < 3:
        return "коммерчески успешный"
    return "высокоуспешный / блокбастер"


def classify_risk(roi):
    if roi < 0:
        return "высокий"
    if roi < 1:
        return "средний"
    if roi < 3:
        return "низкий"
    return "минимальный"


budget = float(sample_input["budget"])
predicted_profit = predicted_revenue - budget
roi = predicted_profit / budget
payback_ratio = predicted_revenue / budget

report = {
    "title": sample_input.get("title"),
    "predicted_revenue": predicted_revenue,
    "predicted_profit": predicted_profit,
    "roi": roi,
    "payback_ratio": payback_ratio,
    "success_category": classify_success(roi),
    "risk_level": classify_risk(roi),
}

report

## Проверка формул

In [ ]:
checks = {
    "profit_formula_ok": abs(report["predicted_profit"] - (report["predicted_revenue"] - budget)) < 1e-6,
    "roi_formula_ok": abs(report["roi"] - (report["predicted_profit"] / budget)) < 1e-6,
    "payback_formula_ok": abs(report["payback_ratio"] - (report["predicted_revenue"] / budget)) < 1e-6,
    "payback_equals_roi_plus_one": abs(report["payback_ratio"] - (report["roi"] + 1)) < 1e-6,
}

checks

## Выводы

1. Production pipeline загружается из `artifacts/movie_revenue_pipeline.joblib`.
2. `pipeline.predict()` возвращает обычную выручку `gross`, а не логарифм.
3. На основе прогноза рассчитываются прибыль, ROI, коэффициент окупаемости, категория успешности и уровень риска.
4. Этот же pipeline используется FastAPI backend без переобучения модели.

## Дополнение: batch-прогноз для нескольких фильмов

Production pipeline может принимать не только один фильм, но и таблицу из нескольких строк. Это удобно для пакетной аналитики и будущих административных экранов.

In [ ]:
batch_inputs = [
    dict(model_input, budget=5_000_000, genre="Drama", runtime=100, country="Russia"),
    dict(model_input, budget=30_000_000, genre="Comedy", runtime=105, country="United States"),
    dict(model_input, budget=120_000_000, genre="Action", runtime=130, country="United States"),
]

batch_df = pd.DataFrame(batch_inputs)
batch_predictions = pipeline.predict(batch_df)

batch_report = batch_df[["genre", "country", "budget", "runtime"]].copy()
batch_report["predicted_revenue"] = batch_predictions
batch_report["predicted_profit"] = batch_report["predicted_revenue"] - batch_report["budget"]
batch_report["roi"] = batch_report["predicted_profit"] / batch_report["budget"]
batch_report["payback_ratio"] = batch_report["predicted_revenue"] / batch_report["budget"]
batch_report

## Дополнение: проверка бюджета как пользовательского ограничения

Production pipeline технически может посчитать прогноз для очень маленького бюджета, но бизнес-интерфейс ограничивает такие значения. Это важно, потому что модель машинного обучения не должна использоваться далеко за пределами обучающего распределения.

In [ ]:
invalid_budget_example = dict(model_input)
invalid_budget_example["budget"] = 1

# Эта ячейка демонстрирует проблему только аналитически.
# В реальном backend такой запрос должен быть отклонён Pydantic-валидацией.
raw_prediction_for_invalid_budget = float(pipeline.predict(pd.DataFrame([invalid_budget_example]))[0])
raw_prediction_for_invalid_budget

Если значение выше выглядит правдоподобно в долларах, ROI всё равно будет бессмысленным из-за бюджета в 1 доллар. Поэтому пользовательская и API-валидация являются частью корректной production-интеграции модели.